# Healthcare Data Quality Assessment: Symptom-Based Disease Classification

**Objective:** Audit data quality and examine how duplicate records and train/test overlap affect interpretation of exploratory machine-learning results.

**Demonstrated skills:** Python/Pandas data cleaning, schema validation, duplicate detection, exploratory analysis, reproducible model evaluation, and transparent reporting.

**Dataset:** Educational symptom–diagnosis CSV files commonly shared on Kaggle. The exact original source, generation process, and redistribution license have **not been independently verified**. The files must be supplied by the person running this notebook; do not redistribute the source CSVs without checking their license.

**Scope:** Educational data analysis only. The model is **not** clinically validated and must not be used for diagnosis or treatment. Repeated rows are not assumed to represent independent patients.

## 1. Setup and data loading

The CSVs must be named `Training.csv` and `Testing.csv`. Place them next to the notebook, in a `data/` subfolder, or upload them into Google Colab. We retain raw copies before any cleaning. `RANDOM_STATE` makes random splits and model fitting reproducible.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

RANDOM_STATE = 42
OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)

def find_csv(filename):
    candidates = [Path(filename), Path("data") / filename, Path("/content") / filename]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Missing {filename}. Upload it to Colab or put it beside the notebook / in data/."
    )

TRAIN_PATH = find_csv("Training.csv")
TEST_PATH = find_csv("Testing.csv")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
print("Raw training shape:", train_raw.shape)
print("Raw testing shape:", test_raw.shape)

### 1.1 Consistent column names and label validation

Only remove completely empty auxiliary `Unnamed:` columns; strip whitespace from column names and diagnosis labels. **Check missing diagnoses before converting values to strings**: otherwise missing labels could become the literal text `"nan"`. The function returns a new DataFrame, leaving the raw one intact.

In [ ]:
def prepare_schema(frame):
    frame = frame.copy()
    frame.columns = frame.columns.str.strip()
    if frame.columns.duplicated().any():
        raise ValueError("Duplicate column names after whitespace normalization")
    empty_unnamed = [
        c for c in frame.columns
        if c.startswith("Unnamed:") and frame[c].isna().all()
    ]
    frame = frame.drop(columns=empty_unnamed)
    if "prognosis" not in frame.columns:
        raise ValueError("Missing required diagnosis column: prognosis")
    if frame["prognosis"].isna().any():
        raise ValueError("Missing diagnosis labels detected")
    frame["prognosis"] = frame["prognosis"].astype("string").str.strip()
    if frame["prognosis"].eq("").any():
        raise ValueError("Blank diagnosis labels detected")
    return frame

train = prepare_schema(train_raw)
test = prepare_schema(test_raw)
features = [c for c in train.columns if c != "prognosis"]
if not features:
    raise ValueError("No symptom columns found")
if set(features) != set(test.columns) - {"prognosis"}:
    raise ValueError("Training and testing symptom columns do not match")
test = test[train.columns]  # Match training column order
for name, frame in [("training", train), ("testing", test)]:
    if frame[features].isna().any().any():
        raise ValueError(f"Missing symptom values in {name}")
    if not frame[features].isin([0, 1]).all().all():
        raise ValueError(f"Non-binary symptom values in {name}")
print("Cleaned train/test shapes:", train.shape, test.shape)
print("Number of diagnoses:", train["prognosis"].nunique())
display(train.head())

## 2. Data quality audit

We distinguish **full-record duplicates** (same symptoms and diagnosis) from **symptom-pattern duplicates** (same symptoms, potentially different diagnoses). Repeated records can reflect legitimate independent observations or artificial repetition; this dataset lacks the provenance needed to decide. Therefore, we **retain the raw rows for descriptive analysis**, and use unique patterns only for a separate sensitivity/evaluation exercise.

In [ ]:
quality_summary = pd.DataFrame({
    "dataset": ["training", "testing"],
    "rows": [len(train), len(test)],
    "missing_cells": [int(train.isna().sum().sum()), int(test.isna().sum().sum())],
    "exact_duplicate_rows": [int(train.duplicated().sum()), int(test.duplicated().sum())],
    "unique_full_records": [len(train.drop_duplicates()), len(test.drop_duplicates())],
    "unique_symptom_patterns": [len(train[features].drop_duplicates()), len(test[features].drop_duplicates())],
    "diagnosis_classes": [train["prognosis"].nunique(), test["prognosis"].nunique()],
})
display(quality_summary)

conflicting_patterns = train.groupby(features, dropna=False)["prognosis"].nunique()
print("Symptom patterns linked to multiple diagnoses:", int((conflicting_patterns > 1).sum()))

train_unique = train.drop_duplicates().reset_index(drop=True)
test_unique = test.drop_duplicates().reset_index(drop=True)
train_pattern_keys = pd.MultiIndex.from_frame(train_unique[features])
test_pattern_keys = pd.MultiIndex.from_frame(test_unique[features])
pattern_overlap = test_pattern_keys.isin(train_pattern_keys)
train_full_keys = pd.MultiIndex.from_frame(train_unique)
test_full_keys = pd.MultiIndex.from_frame(test_unique)
exact_overlap = test_full_keys.isin(train_full_keys)
print(f"Test symptom patterns found in training: {pattern_overlap.sum()} / {len(test_unique)}")
print(f"Exact test records found in training: {exact_overlap.sum()} / {len(test_unique)}")

## 3. Visual storytelling: data quality and symptom patterns

These visualizations are designed for a **client-facing data-quality report**. Every percentage has a clearly defined denominator. They describe an educational dataset, **not patient prevalence or clinical probabilities**.

All figures are exported as high-resolution **PNG** and vector **PDF** files to `figures/`. At the end, a ZIP containing the figures is created; in Google Colab the browser download starts automatically. If your browser blocks it, use the link displayed by Colab's file pane.

The record-frequency chart uses **distinct symptom/diagnosis combinations** to avoid treating unverified repetitions as independent patients. The supplied test overlap is reported separately from the exploratory holdout evaluation.

In [ ]:
# Consistent design system for readable, reusable client deliverables
from matplotlib.ticker import PercentFormatter, MaxNLocator
from textwrap import fill
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink

NAVY = "#142B45"
TEAL = "#008F88"
CORAL = "#E17B68"
BLUE = "#5688C7"
PALE = "#E9F2F5"
GRAY = "#627487"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 16, "axes.titleweight": "bold", "axes.labelcolor": NAVY,
    "text.color": NAVY, "axes.edgecolor": "#D8E2EA",
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "grid.color": "#E7EDF2",
})

def save_chart(fig, filename):
    # Save both a crisp image for slides and a vector PDF for print or proposals.
    fig.savefig(OUTPUT_DIR / f"{filename}.png", dpi=240, bbox_inches="tight", pad_inches=0.28)
    fig.savefig(OUTPUT_DIR / f"{filename}.pdf", bbox_inches="tight", pad_inches=0.28)
    plt.show()
    plt.close(fig)

def chart_header(fig, title, subtitle):
    fig.suptitle(title, x=0.07, y=0.98, ha="left", fontsize=19, fontweight="bold", color=NAVY)
    fig.text(0.07, 0.91, subtitle, ha="left", fontsize=10, color=GRAY)

def clean_axis(ax, grid="x"):
    ax.set_axisbelow(True)
    ax.grid(axis=grid, alpha=0.95)
    ax.tick_params(length=0, pad=8)
    for spine in ax.spines.values():
        spine.set_visible(False)

# Chart 1: training dataset repetition (denominator: all training rows)
n_train = len(train)
n_unique = len(train_unique)
n_repeat = int(train.duplicated().sum())
fig, ax = plt.subplots(figsize=(11, 4.4))
fig.subplots_adjust(left=0.10, right=0.94, top=0.76, bottom=0.24)
chart_header(fig, "Training data | repeated vs distinct", "Share of all training rows · exact full-record matches")
left = 0
for value, color, label in [(n_unique, TEAL, "Distinct records"), (n_repeat, CORAL, "Repeated rows")]:
    pct = 100 * value / n_train
    ax.barh([0], [value], left=left, color=color, height=0.44, edgecolor="white", linewidth=2)
    if pct > 12:
        ax.text(left + value/2, 0, f"{pct:.1f}%\n{value:,} rows", ha="center", va="center", color="white", fontweight="bold", fontsize=14)
    left += value
ax.set_xlim(0, n_train); ax.set_ylim(-0.6, 0.65); ax.set_yticks([])
ax.set_xlabel("Training rows (total = {:,})".format(n_train))
ax.xaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
clean_axis(ax)
fig.text(0.10, 0.10, "■ Distinct symptom + diagnosis records", color=TEAL, fontsize=10)
fig.text(0.56, 0.10, "■ Additional exact repetitions", color=CORAL, fontsize=10)
save_chart(fig, "01_training_duplication")

# Chart 2: provided testing overlap (denominator: distinct supplied test rows)
n_test_unique = len(test_unique)
n_overlap = int(exact_overlap.sum())
n_not_overlap = n_test_unique - n_overlap
fig, ax = plt.subplots(figsize=(11, 4.4))
fig.subplots_adjust(left=0.10, right=0.94, top=0.76, bottom=0.24)
chart_header(fig, "Provided test | exact training overlap", "Share of distinct test records · same symptoms AND diagnosis")
left = 0
for value, color in [(n_overlap, CORAL), (n_not_overlap, TEAL)]:
    ax.barh([0], [value], left=left, color=color, height=0.44, edgecolor="white", linewidth=2)
    pct = 100 * value / n_test_unique
    if pct > 12:
        ax.text(left + value/2, 0, f"{pct:.1f}%\n{value:,} records", ha="center", va="center", color="white", fontweight="bold", fontsize=14)
    left += value
ax.set_xlim(0, n_test_unique); ax.set_ylim(-0.6, 0.65); ax.set_yticks([])
ax.set_xlabel("Distinct supplied test records (total = {:,})".format(n_test_unique))
ax.xaxis.set_major_locator(MaxNLocator(nbins=7, integer=True))
clean_axis(ax)
fig.text(0.10, 0.10, "■ Already present in training", color=CORAL, fontsize=10)
fig.text(0.56, 0.10, f"■ Not present: {n_not_overlap:,} ({100*n_not_overlap/n_test_unique:.1f}%)", color=TEAL, fontsize=10)
save_chart(fig, "02_test_overlap")

# Chart 3: symptom frequency among DISTINCT full records (not population prevalence)
symptom_frequency = train_unique[features].astype(int).mean().sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(11, 7))
fig.subplots_adjust(left=0.28, right=0.91, top=0.82, bottom=0.13)
chart_header(fig, "Most frequent recorded symptoms", "Top 12 symptoms · % of distinct training records with symptom present")
values = symptom_frequency.iloc[::-1] * 100
labels = [fill(s.replace("_", " "), 24) for s in values.index]
ax.barh(range(len(values)), values, color=TEAL, height=0.65)
ax.set_yticks(range(len(values)), labels=labels)
ax.set_xlim(0, max(10, values.max()*1.17))
ax.set_xlabel("Distinct training records with symptom present (%)")
ax.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
for i, v in enumerate(values):
    ax.text(v+0.7, i, f"{v:.1f}%", va="center", fontsize=10, fontweight="bold", color=NAVY)
clean_axis(ax)
fig.text(0.28, 0.055, f"Base: {len(train_unique):,} distinct full records; values are not clinical prevalence.", color=GRAY, fontsize=9)
save_chart(fig, "03_top_symptoms")

# Chart 4: symptom count per distinct record (percentage distribution)
active_symptoms = train_unique[features].astype(int).sum(axis=1)
counts = active_symptoms.value_counts().sort_index()
shares = counts / counts.sum() * 100
fig, ax = plt.subplots(figsize=(11, 5.3))
fig.subplots_adjust(left=0.10, right=0.94, top=0.79, bottom=0.19)
chart_header(fig, "How many symptoms appear per record?", "Distribution of active symptom counts · distinct training records")
ax.bar(shares.index, shares.values, color=BLUE, width=0.75)
ax.set_xlabel("Number of symptoms marked present (1)")
ax.set_ylabel("Share of distinct records (%)")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylim(0, max(shares.values)*1.20)
for x, v in shares.items():
    if v >= 3:
        ax.text(x, v+0.7, f"{v:.1f}%", ha="center", fontsize=9, fontweight="bold", color=NAVY)
clean_axis(ax, grid="y")
fig.text(0.10, 0.06, f"Base: {len(train_unique):,} distinct full records. A count is not a disease severity score.", color=GRAY, fontsize=9)
save_chart(fig, "04_symptom_count_distribution")

# Diagnostic counts are kept as a table rather than a 41-label unreadable chart.
raw_counts = train["prognosis"].value_counts().sort_index()
unique_counts = train_unique["prognosis"].value_counts().reindex(raw_counts.index, fill_value=0)
class_comparison = pd.DataFrame({"raw_rows": raw_counts, "distinct_rows": unique_counts})
print("Diagnosis counts before / after deduplication (first 10 rows):")
display(class_comparison.head(10))
print("Distinct-record range per diagnosis:", unique_counts.min(), "to", unique_counts.max())

## 4. Exploratory machine-learning validation

**Comparison A:** Fit a Random Forest on all supplied training rows and score the supplied test set. Because test records overlap training, this score is **contaminated and not a generalization estimate**.

**Comparison B:** Drop exact repeated full rows, then make a stratified split. This tests generalization to *unseen exact symptom patterns within this artificial dataset* and prevents identical symptom patterns from appearing on both sides. It does **not** establish real-world diagnostic validity, and small per-class sample sizes make estimates unstable. Deduplication here is a deliberate evaluation choice, **not a recommendation to discard repeated real-patient observations**.

In [ ]:
X_raw = train[features]
y_raw = train["prognosis"]
original_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
original_model.fit(X_raw, y_raw)
original_predictions = original_model.predict(test[features])
original_accuracy = accuracy_score(test["prognosis"], original_predictions)
print("PROVIDED TEST SET — OVERLAPS TRAINING; NOT AN INDEPENDENT EVALUATION")
print("Accuracy:", round(original_accuracy, 4))
print("Exact overlapping test records:", int(exact_overlap.sum()), "/", len(test_unique))

X = train_unique[features]
y = train_unique["prognosis"]
min_per_class = int(y.value_counts().min())
print("Minimum unique full records per diagnosis:", min_per_class)
if min_per_class < 2:
    raise ValueError("Stratified splitting requires at least 2 unique rows per diagnosis")

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
train_keys = pd.MultiIndex.from_frame(X_train)
holdout_keys = pd.MultiIndex.from_frame(X_holdout)
if holdout_keys.isin(train_keys).any():
    raise ValueError("Symptom-pattern leakage remains after the split")

model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_holdout)
holdout_accuracy = accuracy_score(y_holdout, pred)
holdout_balanced_accuracy = balanced_accuracy_score(y_holdout, pred)
print("\nDEDUPLICATED STRATIFIED HOLDOUT — EXPLORATORY ONLY")
print("Holdout size:", len(X_holdout), "of", len(X))
print("Accuracy:", round(holdout_accuracy, 4))
print("Balanced accuracy:", round(holdout_balanced_accuracy, 4))
print(classification_report(y_holdout, pred, zero_division=0))

metrics_summary = pd.DataFrame({
    "evaluation": ["Provided test (contaminated)", "Deduplicated holdout (exploratory)"],
    "accuracy": [original_accuracy, holdout_accuracy],
    "balanced_accuracy": [np.nan, holdout_balanced_accuracy],
})
display(metrics_summary)
# Chart 5: model scores with explicit caveats; NOT a clinical validation.
fig, ax = plt.subplots(figsize=(11, 4.8))
fig.subplots_adjust(left=0.31, right=0.91, top=0.76, bottom=0.19)
chart_header(fig, "Model accuracy | evaluation context matters",
             "Two different test protocols · scores are not comparable evidence of clinical performance")
labels = ["Provided test\n(overlaps training)", "Deduplicated holdout\n(exploratory)"]
scores = [original_accuracy * 100, holdout_accuracy * 100]
colors = [CORAL, TEAL]
for i, (score, color) in enumerate(zip(scores, colors)):
    ax.barh(i, score, color=color, height=0.48)
    ax.text(min(score + 1.2, 101), i, f"{score:.1f}%", va="center", fontweight="bold", color=NAVY)
ax.set_yticks([0, 1], labels=labels)
ax.invert_yaxis(); ax.set_xlim(0, 107)
ax.set_xlabel("Classification accuracy (%)")
ax.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
clean_axis(ax)
fig.text(0.31, 0.06, "The supplied test overlaps training; the holdout is small and educational.",
         fontsize=9, color=GRAY)
save_chart(fig, "05_model_evaluation_context")

# Package every generated PNG and PDF for a single-click download.
archive_path = OUTPUT_DIR / "healthcare_data_quality_visualizations.zip"
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for file_path in sorted(OUTPUT_DIR.glob("*.png")) + sorted(OUTPUT_DIR.glob("*.pdf")):
        archive.write(file_path, arcname=file_path.name)
print(f"\nSaved {len(list(OUTPUT_DIR.glob('*.png')))} PNG charts and "
      f"{len(list(OUTPUT_DIR.glob('*.pdf')))} PDF charts in {OUTPUT_DIR.resolve()}")
print("Download all figures:")
display(FileLink(str(archive_path)))
try:
    from google.colab import files as colab_files
    colab_files.download(str(archive_path))
except ImportError:
    print("Outside Colab: open the ZIP using the link above.")


## 5. Interpretation and limitations

- **Chart 1** shows the proportion of repeated rows within the training file. Repetitions are not proof of repeated real patients; their provenance is unverified.
- **Chart 2** measures exact overlap of symptom-and-diagnosis records between the supplied training and test files. Consequently, accuracy on that test file is not an independent estimate of generalization.
- **Charts 3–4** describe the **distinct full-record combinations**, not the prevalence of symptoms in a population. Removing repetitions changes the denominator and therefore the interpretation.
- **Chart 5** puts two exploratory accuracy scores in context. They come from different evaluation protocols and **must not** be sold as validated diagnostic performance.
- **Scope:** A reusable data-quality audit and explanatory visualizations. The dataset's original license and clinical representativeness remain unverified. Do not present this as a clinical decision-support system.

**Deliverables:** Five labeled, high-resolution PNG figures, five vector PDF figures, and a ZIP archive. In Google Colab the ZIP is offered for download automatically after the last cell finishes.